In [ ]:
from pathlib import Path
import subprocess,sys
Path('/kaggle/working/base62.py').write_text('"""Private Kaggle workers. Row-local transform shared by both GBDT families."""\nimport gc\nimport json\nfrom pathlib import Path\nimport sys\nimport time\n\nimport numpy as np\nimport polars as pl\nimport pyarrow.parquet as pq\n\nCUTOFF=np.datetime64(\'2026-10-28T08:06:52.397\'); PAIRS=list(zip(range(26,33),range(40,47)))+[(32,43),(30,46),(31,44)]\nCATS=[5,6,7,8,9,10,11,12,13,14,16,17,18,19,20,21,22,23,25];OUT=Path(\'/kaggle/working/t062\');CACHE=Path(\'/kaggle/temp/t062\')\n\n\ndef numeric(schema):return [\'feature3\']+[c for c,t in schema.items() if c.startswith(\'feature\') and c not in (\'feature1\',\'feature3\') and str(t) in (\'Float32\',\'Int64\',\'Int8\')]\n\n\ndef transform(f,names):\n    x=np.empty((len(f),len(names)+len(PAIRS)),np.float32)\n    for j,c in enumerate(names):x[:,j]=f[c].to_numpy()\n    def unit(k):\n        a=f[f\'feature{k}\'].fill_null([0.]*32).list.to_array(32).to_numpy().astype(\'f4\');return a/np.maximum(np.linalg.norm(a,axis=1,keepdims=True),1e-6)\n    for j,(u,v) in enumerate(PAIRS):x[:,len(names)+j]=(unit(u)*unit(v)).sum(1)\n    return x\n\n\ndef prepare(data):\n    names=numeric(pl.scan_parquet(data/\'train.parquet\').collect_schema());fields=names+list(dict.fromkeys(f\'feature{k}\' for pair in PAIRS for k in pair));assert len(names)+len(PAIRS)==594 and \'request_id\' not in fields\n    for split,n in [(\'train\',1537638),(\'test\',1037090)]:\n        raw=pq.ParquetFile(data/f\'{split}.parquet\');assert raw.metadata.num_rows==n\n        x=np.lib.format.open_memmap(CACHE/f\'{split}.npy\',mode=\'w+\',dtype=\'f4\',shape=(n,594));off=0\n        for batch in raw.iter_batches(columns=fields,batch_size=8192):\n            f=pl.from_arrow(batch);rows=transform(f,names);x[off:off+len(f)]=rows\n            if off==0:np.testing.assert_array_equal(rows[:8],np.vstack([transform(f.slice(i,1),names) for i in range(8)]))\n            off+=len(f)\n        assert off==n;x.flush();del x\n    meta=pl.read_parquet(data/\'train.parquet\',columns=[\'feature1\',\'watched\']);hold=meta[\'feature1\'].to_numpy()>=CUTOFF;assert hold.sum()==307528\n    np.save(CACHE/\'hold.npy\',hold);np.save(CACHE/\'y.npy\',meta[\'watched\'].to_numpy());print(\'prepared 594 row-only fields\',flush=True)\n\n\ndef save_valid(name,ix,y,p):\n    from sklearn.metrics import roc_auc_score\n    assert np.isfinite(p).all() and np.all((p>=0)&(p<=1));np.savez_compressed(OUT/f\'{name}-holdout.npz\',row_ids=ix,y_true=y[ix],probabilities=p);print(name,\'holdout\',roc_auc_score(y[ix],p),flush=True)\n\n\ndef xgb_run(data,name,full):\n    import cupy as cp\n    import xgboost as xgb\n    x=np.load(CACHE/\'train.npy\',mmap_mode=\'r\');y=np.load(CACHE/\'y.npy\');h=np.load(CACHE/\'hold.npy\');ix=np.arange(len(y)) if full else np.flatnonzero(~h); val=np.flatnonzero(h)\n    class Iterator(xgb.DataIter):\n        def __init__(self):self.reset();super().__init__()\n        def reset(self):self.pos=0\n        def next(self,input_data):\n            if self.pos>=len(ix):return False\n            rows=ix[self.pos:self.pos+8192];self.pos+=8192;input_data(data=cp.asarray(x[rows]),label=cp.asarray(y[rows]));return True\n    params=dict(objective=\'binary:logistic\',eval_metric=\'auc\',tree_method=\'hist\',device=\'cuda\',eta=.02,subsample=.8,reg_lambda=10,seed=42,max_bin=127,nthread=4)\n    if name==\'depth12\':params.update(max_depth=12,max_leaves=0,grow_policy=\'depthwise\',colsample_bytree=.8,colsample_bynode=.4,min_child_weight=100,gamma=1)\n    else:params.update(max_depth=0,max_leaves=511,grow_policy=\'lossguide\',colsample_bytree=.3,min_child_weight=200,reg_alpha=1)\n    fit=xgb.QuantileDMatrix(Iterator(),max_bin=127,nthread=4);model=xgb.train(params,fit,900);del fit;gc.collect();cp.get_default_memory_pool().free_all_blocks()\n    arr=np.load(CACHE/\'test.npy\',mmap_mode=\'r\') if full else x; ids=np.arange(len(arr)) if full else val\n    def predict(rows):return cp.asnumpy(model.inplace_predict(cp.asarray(arr[rows])))\n    p=np.concatenate([predict(ids[s:s+8192]) for s in range(0,len(ids),8192)]);np.testing.assert_allclose(p[:8],np.concatenate([predict(ids[j:j+1]) for j in range(8)]),atol=1e-7,rtol=0)\n    if full:\n        assert len(ix)==1537638 and np.isfinite(p).all() and np.all((0<=p)&(p<=1));model.save_model(OUT/\'depth12-full.ubj\');sample=pl.read_csv(data/\'sample_submission.csv\');assert len(sample)==len(p) and sample[\'id\'].n_unique()==len(p);sample.with_columns(pl.Series(\'watched\',p)).write_csv(OUT/\'submission-full-depth12.csv\');print(\'full fit rows\',len(ix),\'test rows\',len(p),flush=True)\n    else:save_valid(name,val,y,p)\n\n\ndef cat_run(data):\n    import pandas as pd\n    from catboost import CatBoostClassifier,Pool\n    x=np.load(CACHE/\'train.npy\',mmap_mode=\'r\');y=np.load(CACHE/\'y.npy\');h=np.load(CACHE/\'hold.npy\');names=[f\'feature{k}\' for k in CATS];cats=pl.read_parquet(data/\'train.parquet\',columns=names).to_pandas()\n    def pool(ix):\n        f=pd.DataFrame(x[ix],columns=[f\'n{k}\' for k in range(x.shape[1])]);\n        for c in names:f[c]=cats[c].iloc[ix].fillna(\'__NA__\').to_numpy()\n        return Pool(f,label=y[ix],cat_features=names,thread_count=4)\n    tr=np.flatnonzero(~h);va=np.flatnonzero(h);fit=pool(tr);valid=pool(va);model=CatBoostClassifier(iterations=3000,depth=8,learning_rate=.06,l2_leaf_reg=10,border_count=128,eval_metric=\'AUC\',task_type=\'GPU\',devices=\'0\',gpu_ram_part=.85,max_ctr_complexity=2,random_seed=42,od_type=\'Iter\',od_wait=300,verbose=250);model.fit(fit,eval_set=valid,use_best_model=True);p=model.predict_proba(valid)[:,1];np.testing.assert_allclose(model.predict_proba(pool(va[:1]))[:,1],p[:1],atol=1e-7);save_valid(\'catboost-ids\',va,y,p);print(\'best trees\',model.tree_count_,flush=True)\n\n\nif __name__==\'__main__\':\n    stage=sys.argv[1];OUT.mkdir(exist_ok=True,parents=True);CACHE.mkdir(exist_ok=True,parents=True);data=next(p.parent for p in Path(\'/kaggle/input\').rglob(\'train.parquet\') if (p.parent/\'sample_submission.csv\').is_file());t=time.monotonic()\n    if stage==\'prepare\':prepare(data)\n    elif stage==\'catboost-ids\':cat_run(data)\n    else:xgb_run(data,stage.split(\'-\')[0],stage.endswith(\'-full\'))\n    (OUT/f\'{stage}.json\').write_text(json.dumps({\'stage\':stage,\'seconds\':time.monotonic()-t,\'full_train_rows\':1537638 if stage.endswith(\'-full\') else None})+\'\\n\')\n')
Path('/kaggle/working/rich63.py').write_text('"""T063: paired depth / richer same-row embedding experiment, two chronological folds."""\nimport json\nfrom pathlib import Path\nimport sys\nimport time\nimport numpy as np\nimport polars as pl\nimport pyarrow.parquet as pq\nfrom sklearn.metrics import roc_auc_score\nimport base62 as base\n\nOUT=Path(\'/kaggle/working/t063\');CACHE=Path(\'/kaggle/temp/t063\');EXTRA=448+49+14\n\n\ndef rich(f):\n    vec=np.stack([f[f\'feature{i}\'].fill_null([0.]*32).list.to_array(32).to_numpy() for i in [*range(26,33),*range(40,47)]],axis=1).astype(\'f4\');bias=np.column_stack([f[f\'feature{i}\'].to_numpy() for i in [*range(33,40),*range(47,54)]])\n    # Bias interpretation is a hypothesis, not a supplied field definition.\n    interactions=np.einsum(\'bik,bjk->bij\',vec[:,:7],vec[:,7:])+bias[:,:7,None]+bias[:,None,7:];return np.column_stack([vec.reshape(len(f),448),interactions.reshape(len(f),49),np.linalg.norm(vec,axis=2)])\n\n\ndef prepare(data):\n    raw=data/\'train.parquet\';names=base.numeric(pl.scan_parquet(raw).collect_schema());fields=names+[f\'feature{i}\' for i in [*range(26,33),*range(40,47)]];n=pq.ParquetFile(raw).metadata.num_rows;assert n==1537638 and len(names)==584 and len(set(fields))==len(fields)\n    m=np.lib.format.open_memmap(CACHE/\'x.npy\',mode=\'w+\',dtype=\'f4\',shape=(n,594+EXTRA));off=0\n    for batch in pq.ParquetFile(raw).iter_batches(columns=fields,batch_size=8192):\n        f=pl.from_arrow(batch);x=np.column_stack([base.transform(f,names),rich(f)]);m[off:off+len(f)]=x;off+=len(f)\n    assert off==n;m.flush();f=pl.read_parquet(raw,columns=[\'feature1\',\'watched\']);np.save(CACHE/\'t.npy\',f[\'feature1\'].to_numpy());np.save(CACHE/\'y.npy\',f[\'watched\'].to_numpy())\n\n\ndef fit(fold,depth,more):\n    import cupy as cp\n    import xgboost as xgb\n    t=np.load(CACHE/\'t.npy\');y=np.load(CACHE/\'y.npy\');x=np.load(CACHE/\'x.npy\',mmap_mode=\'r\');cut=base.CUTOFF if fold==\'primary\' else np.datetime64(\'2026-10-27T08:06:52.397\');tr=np.flatnonzero(t<cut);va=np.flatnonzero((t>=cut)&((fold==\'primary\')|(t<np.datetime64(\'2026-10-27T18:00:00\')))); assert (len(tr),len(va))==((1230110,307528) if fold==\'primary\' else (710330,299969)); width=x.shape[1] if more else 594\n    class Batch(xgb.DataIter):\n        def __init__(self):self.reset();super().__init__()\n        def reset(self):self.i=0\n        def next(self,input_data):\n            if self.i>=len(tr):return False\n            j=tr[self.i:self.i+8192];self.i+=8192;input_data(data=cp.asarray(x[j,:width]),label=cp.asarray(y[j]));return True\n    params=dict(objective=\'binary:logistic\',tree_method=\'hist\',device=\'cuda\',max_depth=depth,eta=.02,subsample=.8,reg_lambda=10,seed=42,max_bin=127,colsample_bytree=.8,colsample_bynode=.4,min_child_weight=100,gamma=1,nthread=4);dm=xgb.QuantileDMatrix(Batch(),max_bin=127,nthread=4);model=xgb.train(params,dm,900);del dm\n    def pred(j):return cp.asnumpy(model.inplace_predict(cp.asarray(x[j,:width])))\n    p=np.concatenate([pred(va[i:i+8192]) for i in range(0,len(va),8192)]);np.testing.assert_array_equal(p[:16],pred(va[:16][::-1])[::-1]);np.testing.assert_allclose(p[:8],np.concatenate([pred(j[None]) for j in va[:8]]),atol=1e-7);assert np.isfinite(p).all();key=f\'{fold}-d{depth}-r{more}\';np.savez_compressed(OUT/f\'{key}.npz\',row_ids=va,y_true=y[va],probabilities=p);return {\'auc\':roc_auc_score(y[va],p),\'fit\':len(tr),\'valid\':len(va),\'width\':width}\n\n\nif __name__==\'__main__\':\n    OUT.mkdir(exist_ok=True,parents=True);CACHE.mkdir(exist_ok=True,parents=True);s=sys.argv[1:];start=time.monotonic()\n    if s==[\'prepare\']:prepare(next(p.parent for p in Path(\'/kaggle/input\').rglob(\'train.parquet\') if (p.parent/\'sample_submission.csv\').exists()));report={}\n    else:report=fit(s[0],int(s[1]),int(s[2]))\n    report[\'seconds\']=time.monotonic()-start;(OUT/(\'-\'.join(s)+\'.json\')).write_text(json.dumps(report)+\'\\n\');print(report,flush=True)\n')
Path('/kaggle/working/t069.py').write_text('"""T069 row-only duration interactions x training budget, fixed rich depth-6 backbone."""\nimport sys,time,json,gc\nfrom pathlib import Path\nimport numpy as np\nimport polars as pl\nimport pyarrow.parquet as pq\nimport base62 as base\nfrom rich63 import rich\n\nOUT=Path(\'/kaggle/working/t069\'); CACHE=Path(\'/kaggle/temp/t069\'); H=[476,477,479,480,483,484,486,487,492,493,500,501,502,503,512,513,514,515];BASE_WIDTH=1105\n\ndef duration(f):\n    d=f[\'feature54\'].to_numpy().astype(\'d\');s=np.column_stack([f[f\'feature{i}\'].to_numpy() for i in H]).astype(\'d\');d=np.where(d>0,d,np.nan);s=np.where(s>=0,s,np.nan)\n    return np.column_stack([np.log1p(s)-np.log1p(d[:,None]),s-d[:,None]]).astype(\'f4\')\n\ndef prepare(raw):\n    names=base.numeric(pl.scan_parquet(raw).collect_schema());fields=names+[f\'feature{i}\' for i in [*range(26,33),*range(40,47)]];assert len(names)==584 and \'request_id\' not in fields\n    pf=pq.ParquetFile(raw);assert pf.metadata.num_rows==1537638;x=np.lib.format.open_memmap(CACHE/\'x.npy\',mode=\'w+\',dtype=\'f4\',shape=(1537638,BASE_WIDTH+2*len(H)));off=0\n    for batch in pf.iter_batches(columns=fields,batch_size=8192):\n        f=pl.from_arrow(batch);x[off:off+len(f)]=np.column_stack([base.transform(f,names),rich(f),duration(f)]);off+=len(f)\n    assert off==len(x);x.flush();f=pl.read_parquet(raw,columns=[\'feature1\',\'watched\']);np.save(CACHE/\'t.npy\',f[\'feature1\'].to_numpy());np.save(CACHE/\'y.npy\',f[\'watched\'].to_numpy())\n\ndef fit(fold,extra):\n    import cupy as cp\n    import xgboost as xgb\n    from sklearn.metrics import roc_auc_score\n    t=np.load(CACHE/\'t.npy\');y=np.load(CACHE/\'y.npy\');x=np.load(CACHE/\'x.npy\',mmap_mode=\'r\');cut=base.CUTOFF if fold==\'primary\' else np.datetime64(\'2026-10-27T08:06:52.397\');tr=np.flatnonzero(t<cut);va=np.flatnonzero((t>=cut)&((fold==\'primary\')|(t<np.datetime64(\'2026-10-27T18:00:00\'))));assert (len(tr),len(va))==((1230110,307528) if fold==\'primary\' else (710330,299969));width=x.shape[1] if extra else BASE_WIDTH\n    class Iter(xgb.DataIter):\n        def __init__(self):self.reset();super().__init__()\n        def reset(self):self.i=0\n        def next(self,input_data):\n            if self.i>=len(tr):return False\n            j=tr[self.i:self.i+8192];self.i+=8192;input_data(data=cp.asarray(x[j,:width]),label=cp.asarray(y[j]));return True\n    params=dict(objective=\'binary:logistic\',tree_method=\'hist\',device=\'cuda\',max_depth=6,eta=.02,subsample=.8,reg_lambda=10,seed=42,max_bin=127,colsample_bytree=.8,colsample_bynode=.4,min_child_weight=100,gamma=1,nthread=4);dm=xgb.QuantileDMatrix(Iter(),max_bin=127,nthread=4);model=xgb.train(params,dm,2400);del dm;gc.collect();cp.get_default_memory_pool().free_all_blocks()\n    for rounds in [900,1600,2400]:\n        def pred(j):return cp.asnumpy(model.inplace_predict(cp.asarray(x[j,:width]),iteration_range=(0,rounds)))\n        p=np.concatenate([pred(va[i:i+8192]) for i in range(0,len(va),8192)]);np.testing.assert_allclose(p[:8],np.concatenate([pred(j[None]) for j in va[:8]]),atol=1e-7);assert np.isfinite(p).all();key=f\'{fold}-{extra}-{rounds}\';np.savez_compressed(OUT/f\'{key}.npz\',row_ids=va,y_true=y[va],probabilities=p);print(key,roc_auc_score(y[va],p),flush=True)\n\nif __name__==\'__main__\':\n    OUT.mkdir(exist_ok=True,parents=True);CACHE.mkdir(exist_ok=True,parents=True);args=sys.argv[1:];start=time.monotonic()\n    if args==[\'prepare\']:prepare(next(p for p in Path(\'/kaggle/input\').rglob(\'train.parquet\') if (p.parent/\'sample_submission.csv\').is_file()))\n    else:fit(args[0],int(args[1]))\n    (OUT/(\'-\'.join(args)+\'.json\')).write_text(json.dumps({\'seconds\':time.monotonic()-start})+\'\\n\')\n')
Path('/kaggle/working/t070.py').write_text('"""Same-row neural models on the exact T069 enriched input. Train-only scaling."""\nfrom pathlib import Path\nimport json,sys,time\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom sklearn.metrics import roc_auc_score\nimport t069\n\nOUT=Path(\'/kaggle/working/t070\');CACHE=t069.CACHE\n\nclass Model(nn.Module):\n    def __init__(self,width,kind):\n        super().__init__();self.kind=kind;self.project=nn.Sequential(nn.Linear(width,192),nn.LayerNorm(192),nn.SiLU());self.blocks=nn.ModuleList(nn.Sequential(nn.LayerNorm(192),nn.Linear(192,384),nn.SiLU(),nn.Dropout(.18),nn.Linear(384,192)) for _ in range(2));self.head=nn.Linear(192,1)\n        if kind==\'gated\':self.gate=nn.Linear(192,192);self.value=nn.Linear(192,192)\n    def forward(self,x):\n        h=self.project(x);initial=h\n        for block in self.blocks:h=h+.5*block(h)\n        if self.kind==\'gated\':h=h+initial*torch.sigmoid(self.gate(h))*self.value(h)\n        return self.head(h).squeeze(-1)\n\ndef fit(fold,kind):\n    assert kind in (\'residual\',\'gated\');torch.manual_seed(42);rng=np.random.default_rng(42);torch.set_num_threads(2);torch.set_float32_matmul_precision(\'high\');device=\'cuda\';t=np.load(CACHE/\'t.npy\');y=np.load(CACHE/\'y.npy\');x=np.load(CACHE/\'x.npy\',mmap_mode=\'r\');cut=t069.base.CUTOFF if fold==\'primary\' else np.datetime64(\'2026-10-27T08:06:52.397\');tr=np.flatnonzero(t<cut);va=np.flatnonzero((t>=cut)&((fold==\'primary\')|(t<np.datetime64(\'2026-10-27T18:00\'))));assert (len(tr),len(va))==((1230110,307528) if fold==\'primary\' else (710330,299969));assert x.shape[1]==1141\n    # Fit a robust location and scale on a fixed sample of training rows only.\n    sample=x[np.sort(rng.choice(tr,32768,replace=False))];q=np.nanpercentile(sample,[10,50,90],axis=0);loc=np.nan_to_num(q[1]).astype(\'f4\');scale=np.nan_to_num((q[2]-q[0])/2.563).astype(\'f4\');sd=np.nanstd(sample,axis=0);scale=np.where(scale>1e-6,scale,np.where(sd>1e-6,sd,1)).astype(\'f4\');del sample\n    def encode(rows):\n        z=(x[rows]-loc)/scale;return np.nan_to_num(np.clip(z,-8,8),nan=0).astype(\'f4\',copy=False)\n    # Cache only this fold on device; avoid repeated random reads of the memmap.\n    def cache(rows):\n        z=torch.empty((len(rows),x.shape[1]),device=device,dtype=torch.float16)\n        for i in range(0,len(rows),4096):z[i:i+4096]=torch.from_numpy(encode(rows[i:i+4096])).to(device=device,dtype=z.dtype)\n        return z\n    train=cache(tr);valid=cache(va);labels=torch.from_numpy(y[tr].astype(\'f4\')).to(device);model=Model(x.shape[1],kind).to(device);nn.init.constant_(model.head.bias,float(np.log(y[tr].mean()/(1-y[tr].mean()))));opt=torch.optim.AdamW(model.parameters(),lr=.00045,weight_decay=.015);metrics={}\n    for epoch in range(1,9):\n        model.train();order=torch.randperm(len(tr),device=device)\n        for j in order.split(2048):\n            opt.zero_grad(set_to_none=True);loss=F.binary_cross_entropy_with_logits(model(train[j].float()),labels[j]);loss.backward();nn.utils.clip_grad_norm_(model.parameters(),2.);opt.step()\n        if epoch==5:\n            for group in opt.param_groups:group[\'lr\']=.00015\n        if epoch in (2,5,8):\n            model.eval()\n            with torch.inference_mode():\n                def predict(z):return model(z.float()).sigmoid().cpu().numpy()\n                p=np.concatenate([predict(z) for z in valid.split(2048)]);np.testing.assert_allclose(p[:12],np.concatenate([predict(valid[i:i+1]) for i in range(12)]),atol=2e-4,rtol=2e-4)\n            assert np.isfinite(p).all();auc=roc_auc_score(y[va],p);metrics[epoch]=auc;np.savez_compressed(OUT/f\'{fold}-{kind}-{epoch}.npz\',row_ids=va,y_true=y[va],probabilities=p);print(fold,kind,epoch,auc,flush=True)\n    return metrics\n\nif __name__==\'__main__\':\n    args=sys.argv[1:];OUT.mkdir(exist_ok=True,parents=True);CACHE.mkdir(exist_ok=True,parents=True);start=time.monotonic()\n    if args==[\'prepare\']:t069.prepare(next(p for p in Path(\'/kaggle/input\').rglob(\'train.parquet\') if (p.parent/\'sample_submission.csv\').exists()));scores={}\n    else:scores=fit(*args)\n    (OUT/(\'-\'.join(args)+\'.json\')).write_text(json.dumps({\'scores\':scores,\'seconds\':time.monotonic()-start})+\'\\n\')\n')
Path('/kaggle/working/t071.py').write_text('"""Incremental categorical information on top of T070\'s numeric gated backbone."""\nimport json,sys,time\nfrom pathlib import Path\nimport numpy as np\nimport polars as pl\nimport torch\nfrom torch import nn\nimport torch.nn.functional as F\nfrom sklearn.metrics import roc_auc_score\nimport t070\n\nOUT=Path(\'/kaggle/working/t071\');CACHE=t070.CACHE\nVIEW=[6,7,8,9,10,11,12,13,14,16,17];CONTENT=[18,19,20,21,22,23,25];COLS=[f\'feature{i}\' for i in VIEW+CONTENT]\n\ndef vocabulary(s):\n    # Fixed cap; deterministic tie break. Zero is reserved for unseen/missing values.\n    c=s.drop_nulls().value_counts(name=\'n\').filter(pl.col(\'n\')>=10).sort([\'n\',s.name],descending=[True,False]).head(8191)\n    return c.select(s.name).with_row_index(\'code\',offset=1)\n\ndef codes(s,tab):return s.to_frame().join(tab,on=s.name,how=\'left\',maintain_order=\'left\',validate=\'m:1\')[\'code\'].fill_null(0).to_numpy().astype(\'int16\')\n\nclass WithCategories(nn.Module):\n    def __init__(self,width,cards,kind):\n        super().__init__();self.numeric=t070.Model(width,\'gated\');self.kind=kind\n        if kind!=\'control\':\n            self.tables=nn.ModuleList(nn.Embedding(n,8,padding_idx=0) for n in cards);dim=8*len(cards)+(len(VIEW)*len(CONTENT) if kind==\'cross\' else 0);self.extra=nn.Sequential(nn.Linear(dim,64),nn.SiLU(),nn.Dropout(.15),nn.Linear(64,1));nn.init.zeros_(self.extra[-1].bias)\n    def forward(self,x,c):\n        y=self.numeric(x)\n        if self.kind==\'control\':return y\n        if self.training:c=c.masked_fill(torch.rand(c.shape,device=c.device)<.12,0)\n        e=torch.stack([tab(c[:,i].long()) for i,tab in enumerate(self.tables)],1);z=e.flatten(1)\n        if self.kind==\'cross\':z=torch.cat([z,(torch.bmm(e[:,:len(VIEW)],e[:,len(VIEW):].transpose(1,2))/(8**.5)).flatten(1)],1)\n        return y+self.extra(z).squeeze(-1)\n\ndef fit(raw,fold,kind):\n    torch.manual_seed(43);torch.set_num_threads(2);torch.set_float32_matmul_precision(\'high\');rng=np.random.default_rng(42);t=np.load(CACHE/\'t.npy\');x=np.load(CACHE/\'x.npy\',mmap_mode=\'r\');y=np.load(CACHE/\'y.npy\');cut=t070.t069.base.CUTOFF if fold==\'primary\' else np.datetime64(\'2026-10-27T08:06:52.397\');tr=np.flatnonzero(t<cut);va=np.flatnonzero((t>=cut)&((fold==\'primary\')|(t<np.datetime64(\'2026-10-27T18:00\'))));assert (len(tr),len(va))==((1230110,307528) if fold==\'primary\' else (710330,299969));assert x.shape[1]==1141\n    f=pl.read_parquet(raw,columns=COLS);cats=[];cards=[]\n    for col in COLS:\n        s=f[col];tab=vocabulary(s.gather(tr));cats.append(codes(s,tab));cards.append(len(tab)+1)\n    c=np.column_stack(cats);del f,cats;sample=x[np.sort(rng.choice(tr,32768,replace=False))];q=np.nanpercentile(sample,[10,50,90],axis=0);loc=np.nan_to_num(q[1]).astype(\'f4\');sc=np.nan_to_num((q[2]-q[0])/2.563).astype(\'f4\');sd=np.nanstd(sample,axis=0);sc=np.where(sc>1e-6,sc,np.where(sd>1e-6,sd,1));del sample\n    def cache(ix):\n        xx=torch.empty((len(ix),x.shape[1]),device=\'cuda\',dtype=torch.float16)\n        for i in range(0,len(ix),4096):xx[i:i+4096]=torch.from_numpy(np.nan_to_num(np.clip((x[ix[i:i+4096]]-loc)/sc,-8,8),nan=0)).to(device=\'cuda\',dtype=xx.dtype)\n        return xx,torch.from_numpy(c[ix]).to(\'cuda\')\n    train,ct=cache(tr);valid,cv=cache(va);labels=torch.from_numpy(y[tr].astype(\'f4\')).to(\'cuda\');model=WithCategories(x.shape[1],cards,kind).cuda();nn.init.constant_(model.numeric.head.bias,float(np.log(y[tr].mean()/(1-y[tr].mean()))));opt=torch.optim.AdamW(model.parameters(),lr=.00045,weight_decay=.015)\n    for epoch in range(1,9):\n        model.train()\n        for j in torch.randperm(len(tr),device=\'cuda\').split(2048):\n            opt.zero_grad(set_to_none=True);F.binary_cross_entropy_with_logits(model(train[j].float(),ct[j]),labels[j]).backward();nn.utils.clip_grad_norm_(model.parameters(),2.);opt.step()\n        if epoch==5:\n            for group in opt.param_groups:group[\'lr\']=.00015\n        if epoch in [5,8]:\n            model.eval()\n            with torch.inference_mode():\n                def pred(z,cat):return model(z.float(),cat).sigmoid().cpu().numpy()\n                p=np.concatenate([pred(z,cat) for z,cat in zip(valid.split(2048),cv.split(2048))]);np.testing.assert_allclose(p[:12],np.concatenate([pred(valid[i:i+1],cv[i:i+1]) for i in range(12)]),atol=2e-4,rtol=2e-4)\n            assert np.isfinite(p).all();np.savez_compressed(OUT/f\'{fold}-{kind}-{epoch}.npz\',row_ids=va,y_true=y[va],probabilities=p);print(fold,kind,epoch,roc_auc_score(y[va],p),flush=True)\n\nif __name__==\'__main__\':\n    OUT.mkdir(exist_ok=True,parents=True);CACHE.mkdir(exist_ok=True,parents=True);raw=next(p for p in Path(\'/kaggle/input\').rglob(\'train.parquet\') if (p.parent/\'sample_submission.csv\').exists());args=sys.argv[1:];start=time.monotonic()\n    if args==[\'prepare\']:t070.t069.prepare(raw)\n    else:fit(raw,*args)\n    (OUT/(\'-\'.join(args)+\'.json\')).write_text(json.dumps({\'seconds\':time.monotonic()-start})+\'\\n\')\n')
for args in [['prepare']]+[[fold,kind] for fold in ['earlier','primary'] for kind in ['control','additive','cross']]:
    subprocess.run([sys.executable,'-u','/kaggle/working/t071.py',*args],check=True)
